In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio

DATASET_DIR = Path("/projects/EEG-foundation-model/datalake/raw/MNE-bnci-data/database/data-sets/001-2014")
FILE_PATH = DATASET_DIR / "A01E.mat"

In [17]:
files = sorted(DATASET_DIR.glob("*.mat"))

print(f"Number of T files: {len(files)}")
print("Files:")
for f in files:
    print(" ", f.name)

Number of T files: 18
Files:
  A01E.mat
  A01T.mat
  A02E.mat
  A02T.mat
  A03E.mat
  A03T.mat
  A04E.mat
  A04T.mat
  A05E.mat
  A05T.mat
  A06E.mat
  A06T.mat
  A07E.mat
  A07T.mat
  A08E.mat
  A08T.mat
  A09E.mat
  A09T.mat


In [16]:
mat = sio.loadmat(FILE_PATH)

print(f"File: {FILE_PATH}\n")

print("=== TOP-LEVEL KEYS ===")
for k, v in mat.items():
    print(f"{k}: type={type(v)}, shape={getattr(v, 'shape', None)}")

if "data" in mat:
    data = mat["data"]
    print("\n=== DATA FIELD ===")
    print(f"type={type(data)}, shape={data.shape}, dtype={data.dtype}")

    try:
        data_struct = data[0, 0]
        print("\n=== DATA STRUCT FIELDS ===")
        print(data_struct.dtype.names)

        for field in data_struct.dtype.names:
            val = data_struct[field]
            print(f"{field}: type={type(val)}, shape={getattr(val, 'shape', None)}")
    except Exception as e:
        print("\n[WARN] Could not parse structured data:", e)

File: /projects/EEG-foundation-model/datalake/raw/MNE-bnci-data/database/data-sets/001-2014/A01T.mat

=== TOP-LEVEL KEYS ===
__header__: type=<class 'bytes'>, shape=None
__version__: type=<class 'str'>, shape=None
__globals__: type=<class 'list'>, shape=None
data: type=<class 'numpy.ndarray'>, shape=(1, 9)

=== DATA FIELD ===
type=<class 'numpy.ndarray'>, shape=(1, 9), dtype=object

=== DATA STRUCT FIELDS ===
('X', 'trial', 'y', 'fs', 'classes', 'artifacts', 'gender', 'age')
X: type=<class 'numpy.ndarray'>, shape=(1, 1)
trial: type=<class 'numpy.ndarray'>, shape=(1, 1)
y: type=<class 'numpy.ndarray'>, shape=(1, 1)
fs: type=<class 'numpy.ndarray'>, shape=(1, 1)
classes: type=<class 'numpy.ndarray'>, shape=(1, 1)
artifacts: type=<class 'numpy.ndarray'>, shape=(1, 1)
gender: type=<class 'numpy.ndarray'>, shape=(1, 1)
age: type=<class 'numpy.ndarray'>, shape=(1, 1)


In [19]:

data = mat["data"]

run_idx = 6
run = data[0, run_idx]

print(f"=== RUN {run_idx} ===\n")

for field in run.dtype.names:
    val = run[field][0, 0]

    if field == "X":
        print(f"{field}: [SKIPPED]")
        continue

    if isinstance(val, np.ndarray):
        print(f"{field}: shape={val.shape}, dtype={val.dtype}")
        
        if val.size < 20:
            print(val.squeeze())
        else:
            print(val.squeeze()[:10], "...")

    else:
        print(f"{field}: {val}")

=== RUN 6 ===

X: [SKIPPED]
trial: shape=(48, 1), dtype=int32
[  251  2254  4172  6124  8132 10243 12160 14210 16141 18139] ...
y: shape=(48, 1), dtype=uint8
[2 1 1 3 3 2 3 4 1 4] ...
fs: shape=(1, 1), dtype=uint8
250
classes: shape=(1, 4), dtype=object
[array(['left hand'], dtype='<U9') array(['right hand'], dtype='<U10')
 array(['feet'], dtype='<U4') array(['tongue'], dtype='<U6')]
artifacts: shape=(48, 1), dtype=uint8
[0 0 0 0 0 0 0 0 0 0] ...
gender: shape=(1,), dtype=<U6
female
age: shape=(1, 1), dtype=uint8
22


In [20]:
for file_path in files:
    mat = sio.loadmat(file_path)
    data = mat["data"]

    print(f"\n=== {file_path.name} ===")

    for run_idx in range(data.shape[1]):
        run = data[0, run_idx]

        trial = run["trial"][0, 0].squeeze()
        y = run["y"][0, 0].squeeze()

        if trial.size > 0:
            print(
                f"run {run_idx} | "
                f"trial shape={run['trial'][0, 0].shape} | "
                f"y shape={run['y'][0, 0].shape} | "
                f"labels={dict(Counter(y.tolist()))}"
            )


=== A01E.mat ===
run 3 | trial shape=(48, 1) | y shape=(48, 1) | labels={1: 12, 2: 12, 3: 12, 4: 12}
run 4 | trial shape=(48, 1) | y shape=(48, 1) | labels={4: 12, 3: 12, 2: 12, 1: 12}
run 5 | trial shape=(48, 1) | y shape=(48, 1) | labels={1: 12, 4: 12, 2: 12, 3: 12}
run 6 | trial shape=(48, 1) | y shape=(48, 1) | labels={2: 12, 4: 12, 1: 12, 3: 12}
run 7 | trial shape=(48, 1) | y shape=(48, 1) | labels={2: 12, 1: 12, 3: 12, 4: 12}
run 8 | trial shape=(48, 1) | y shape=(48, 1) | labels={3: 12, 1: 12, 4: 12, 2: 12}

=== A01T.mat ===
run 3 | trial shape=(48, 1) | y shape=(48, 1) | labels={4: 12, 3: 12, 2: 12, 1: 12}
run 4 | trial shape=(48, 1) | y shape=(48, 1) | labels={1: 12, 4: 12, 2: 12, 3: 12}
run 5 | trial shape=(48, 1) | y shape=(48, 1) | labels={2: 12, 4: 12, 1: 12, 3: 12}
run 6 | trial shape=(48, 1) | y shape=(48, 1) | labels={2: 12, 1: 12, 3: 12, 4: 12}
run 7 | trial shape=(48, 1) | y shape=(48, 1) | labels={3: 12, 1: 12, 4: 12, 2: 12}
run 8 | trial shape=(48, 1) | y shape=(4

In [23]:

from pathlib import Path
from logging import Logger
import random
import re

import mne
import numpy as np
import scipy.io as sio
from abc import ABC, abstractmethod
import mne
from pathlib import Path
from logging import Logger


class PreprocessorModel(ABC):
    def __init__(self, dataset_dir: str | Path, logger: Logger = None):
        self.dataset_dir = Path(dataset_dir)
        self.logger = logger
    
    @abstractmethod
    def get_files(self, ratio: float, seed: int = 42) -> list[Path]:
        """Return a list of file paths in the dataset directory."""
        pass
    
    @abstractmethod
    def load_data(self, file_path: Path) -> mne.io.BaseRaw:
        """Load and return the data from the given file path as an MNE Raw object."""
        pass
    @abstractmethod
    def load_labels(self, file_path: Path, raw: mne.io.BaseRaw = None) -> mne.Annotations:
        """Load and return the labels associated with the given file path."""
        pass
    
    def load(self, file_path: Path) -> tuple[mne.io.BaseRaw, mne.Annotations]:
        """Load and return both the data and labels from the given file path."""
        raw = self.load_data(file_path)
        annotations = self.load_labels(file_path, raw=raw)
        subject_id = self.get_subject_id(file_path)
        return raw, annotations, subject_id
    
    @abstractmethod
    def get_subject_id(self, file_path: Path) -> str:
        """Extract and return the subject ID from the given file path."""
        pass



BCI2A_CHANNEL_LIST = [
    "EEG-Fz",
    "EEG-0", "EEG-1", "EEG-2", "EEG-3", "EEG-4", "EEG-5",
    "EEG-C3", "EEG-6", "EEG-Cz", "EEG-7", "EEG-C4",
    "EEG-8", "EEG-9", "EEG-10", "EEG-11", "EEG-12",
    "EEG-13", "EEG-14", "EEG-Pz", "EEG-15", "EEG-16",
    "EOG-left", "EOG-central", "EOG-right",
]

BCI2A_STAGE_MAP = {
    1: 0,
    2: 1,
    3: 2,
    4: 3,
}


class BCI2A_preprocessor(PreprocessorModel):
    def __init__(self, dataset_dir: str | Path, logger: Logger = None):
        super().__init__(dataset_dir, logger)
        self.logger = logger

    def get_files(self, ratio: float, seed: int = 42) -> list[Path]:
        if not (0.0 < ratio <= 1.0):
            raise ValueError(f"ratio must be in (0, 1], got {ratio}")

        rootdir = Path(self.dataset_dir).expanduser()

        files = sorted(rootdir.glob("A0*[TE].mat"))

        if self.logger:
            self.logger.info(f"Found {len(files)} MAT files in {rootdir}")

        subj_ids = [self.get_subject_id(f) for f in files]
        unique_subj_ids = sorted(set(subj_ids))

        n_subj_total = len(unique_subj_ids)
        n_subj_keep = max(1, int(n_subj_total * ratio))

        rng = random.Random(seed)
        rng.shuffle(unique_subj_ids)
        subj_ids_to_keep = set(unique_subj_ids[:n_subj_keep])

        kept_files = [f for f, sid in zip(files, subj_ids) if sid in subj_ids_to_keep]

        if self.logger:
            self.logger.info(
                f"Subject-wise selection: keeping {n_subj_keep}/{n_subj_total} subjects "
                f"({ratio * 100:.2f}%), resulting in {len(kept_files)}/{len(files)} files"
            )

        return kept_files

    def _valid_runs(self, file_path: Path):
        mat = sio.loadmat(file_path)
        data = mat["data"]

        runs = []
        for run_idx in range(data.shape[1]):
            run = data[0, run_idx]

            X = run["X"][0, 0]
            trial = run["trial"][0, 0].squeeze()
            y = run["y"][0, 0].squeeze()
            fs = int(run["fs"][0, 0].squeeze())

            if trial.size == 0 or y.size == 0:
                continue

            trial = np.atleast_1d(trial).astype(int)
            y = np.atleast_1d(y).astype(int)

            artifacts = run["artifacts"][0, 0].squeeze()
            artifacts = np.atleast_1d(artifacts).astype(int) if artifacts.size > 0 else np.zeros_like(y)

            runs.append(
                {
                    "run_idx": run_idx,
                    "X": X,
                    "trial": trial,
                    "y": y,
                    "fs": fs,
                    "artifacts": artifacts,
                }
            )

        return runs

    def load_data(self, file_path: Path) -> mne.io.BaseRaw:
        runs = self._valid_runs(file_path)

        if len(runs) == 0:
            raise ValueError(f"No valid run with trials found in {file_path}")

        xs = []

        for r in runs:
            X = r["X"]

            if X.ndim != 2:
                raise ValueError(f"Unexpected X shape in {file_path}: {X.shape}")

            if X.shape[1] == len(BCI2A_CHANNEL_LIST):
                X = X.T
            elif X.shape[0] == len(BCI2A_CHANNEL_LIST):
                pass
            else:
                raise ValueError(
                    f"Channel mismatch in {file_path}: X shape={X.shape}, "
                    f"expected {len(BCI2A_CHANNEL_LIST)} channels"
                )

            xs.append(X.astype(np.float64, copy=False))

        x = np.concatenate(xs, axis=1)

        fs = runs[0]["fs"]

        info = mne.create_info(
            ch_names=list(BCI2A_CHANNEL_LIST),
            sfreq=float(fs),
            ch_types=["eeg"] * 22 + ["eog"] * 3,
        )

        return mne.io.RawArray(x, info, verbose=False)

    def load_labels(self, file_path: Path, raw: mne.io.BaseRaw = None) -> mne.Annotations:
        runs = self._valid_runs(file_path)

        if len(runs) == 0:
            raise ValueError(f"No valid run with labels found in {file_path}")

        onsets = []
        durations = []
        descriptions = []

        offset_sec = 0.0

        for r in runs:
            X = r["X"]
            fs = r["fs"]
            n_samples = X.shape[0] if X.shape[1] == len(BCI2A_CHANNEL_LIST) else X.shape[1]

            for start, label, artifact in zip(r["trial"], r["y"], r["artifacts"]):
                if int(artifact) != 0:
                    continue

                label = int(label)

                if label not in BCI2A_STAGE_MAP:
                    raise KeyError(f"Unknown BCI2A label {label} in {file_path}")

                onset = offset_sec + ((int(start) - 1) / float(fs))

                onsets.append(onset)
                durations.append(4.0)
                descriptions.append(str(BCI2A_STAGE_MAP[label]))

            offset_sec += n_samples / float(fs)

        return mne.Annotations(
            onset=onsets,
            duration=durations,
            description=descriptions,
            orig_time=None,
        )

    def get_subject_id(self, file_path: Path) -> str:
        m = re.match(r"^(A0\d)[TE]\.mat$", file_path.name)
        if m:
            return m.group(1)
        return file_path.stem[:3]

In [24]:
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("/projects/EEG-foundation-model/datalake/raw/MNE-bnci-data/database/data-sets/001-2014")

preprocessor = BCI2A_preprocessor(DATASET_DIR)

files = preprocessor.get_files(ratio=1.0)

print("n files:", len(files))
print([f.name for f in files[:5]])

n files: 18
['A01E.mat', 'A01T.mat', 'A02E.mat', 'A02T.mat', 'A03E.mat']


In [25]:
file_path = DATASET_DIR / "A01T.mat"

raw = preprocessor.load_data(file_path)
ann = preprocessor.load_labels(file_path, raw)

print(raw)
print("raw shape:", raw.get_data().shape)
print("sfreq:", raw.info["sfreq"])
print("channels:", raw.ch_names)
print("n annotations:", len(ann))
print("labels:", Counter(ann.description))

print(ann[:10])

<RawArray | 25 x 580410 (2321.6 s), ~110.7 MiB, data loaded>
raw shape: (25, 580410)
sfreq: 250.0
channels: ['EEG-Fz', 'EEG-0', 'EEG-1', 'EEG-2', 'EEG-3', 'EEG-4', 'EEG-5', 'EEG-C3', 'EEG-6', 'EEG-Cz', 'EEG-7', 'EEG-C4', 'EEG-8', 'EEG-9', 'EEG-10', 'EEG-11', 'EEG-12', 'EEG-13', 'EEG-14', 'EEG-Pz', 'EEG-15', 'EEG-16', 'EOG-left', 'EOG-central', 'EOG-right']
n annotations: 273
labels: Counter({np.str_('1'): 69, np.str_('0'): 69, np.str_('2'): 68, np.str_('3'): 67})
<Annotations | 10 segments: 0 (2), 1 (3), 2 (3), 3 (2)>


In [26]:
for f in files:
    try:
        raw = preprocessor.load_data(f)
        ann = preprocessor.load_labels(f, raw)

        print(
            f"{f.name} | "
            f"shape={raw.get_data().shape} | "
            f"sfreq={raw.info['sfreq']} | "
            f"n_ann={len(ann)} | "
            f"labels={dict(Counter(ann.description))}"
        )

    except Exception as e:
        print(f"[ERROR] {f.name}: {e}")

A01E.mat | shape=(25, 580410) | sfreq=250.0 | n_ann=281 | labels={np.str_('0'): 71, np.str_('1'): 70, np.str_('2'): 69, np.str_('3'): 71}
A01T.mat | shape=(25, 580410) | sfreq=250.0 | n_ann=273 | labels={np.str_('3'): 67, np.str_('2'): 68, np.str_('1'): 69, np.str_('0'): 69}
A02E.mat | shape=(25, 580410) | sfreq=250.0 | n_ann=283 | labels={np.str_('0'): 71, np.str_('1'): 71, np.str_('2'): 69, np.str_('3'): 72}
A02T.mat | shape=(25, 580410) | sfreq=250.0 | n_ann=270 | labels={np.str_('0'): 67, np.str_('1'): 69, np.str_('2'): 66, np.str_('3'): 68}
A03E.mat | shape=(25, 580410) | sfreq=250.0 | n_ann=273 | labels={np.str_('0'): 67, np.str_('1'): 70, np.str_('2'): 68, np.str_('3'): 68}
A03T.mat | shape=(25, 580410) | sfreq=250.0 | n_ann=270 | labels={np.str_('0'): 69, np.str_('1'): 68, np.str_('2'): 66, np.str_('3'): 67}
A04E.mat | shape=(25, 580410) | sfreq=250.0 | n_ann=228 | labels={np.str_('3'): 53, np.str_('2'): 59, np.str_('1'): 57, np.str_('0'): 59}
A04T.mat | shape=(25, 580410) | sf